In [26]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
import pywt

In [51]:
import warnings
warnings.filterwarnings("ignore")

In [27]:
face_cascade = cv2.CascadeClassifier('./opencv/haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('./opencv/haarcascade_eye.xml')

In [28]:
def get_cropped_image_if_2_eyes(image_path):

    img = cv2.imread(image_path)

    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray,1.3,5)

    for (x,y,w,h) in faces:

        roi_gray = gray[y:y+h,x:x+w]
        roi_color = img[y:y+h,x:x+w]

        eyes = eye_cascade.detectMultiScale(roi_gray)

        if len(eyes) >= 2:
            return roi_color

In [29]:
path_to_data = "./dataset/"
path_to_cr_data = "./dataset/cropped/"

In [30]:
img_dirs = []

for entry in os.scandir(path_to_data):
    if entry.is_dir():
        img_dirs.append(entry.path)

img_dirs

['./dataset/lionel_messi',
 './dataset/maria_sharapova',
 './dataset/roger_federer',
 './dataset/serena_williams',
 './dataset/virat_kohli']

In [31]:
cropped_image_dirs = []
celebrity_file_names_dict = {}

for img_dir in img_dirs:

    count = 1
    celebrity_name = img_dir.split('/')[-1]
    print(celebrity_name)

    celebrity_file_names_dict[celebrity_name] = []

    for entry in os.scandir(img_dir):

        roi_color = get_cropped_image_if_2_eyes(entry.path)

        if roi_color is not None:

            cropped_folder = path_to_cr_data + celebrity_name

            if not os.path.exists(cropped_folder):
                os.makedirs(cropped_folder)
                cropped_image_dirs.append(cropped_folder)

            cropped_file_name = celebrity_name + str(count) + ".png"
            cropped_file_path = cropped_folder + "/" + cropped_file_name

            cv2.imwrite(cropped_file_path, roi_color)

            celebrity_file_names_dict[celebrity_name].append(cropped_file_path)

            count += 1

lionel_messi
maria_sharapova
roger_federer
serena_williams
virat_kohli


In [32]:
celebrity_file_names_dict = {}

for img_dir in os.scandir(path_to_cr_data):

    celebrity_name = img_dir.path.split('/')[-1]
    file_list = []

    for entry in os.scandir(img_dir.path):
        file_list.append(entry.path)

    celebrity_file_names_dict[celebrity_name] = file_list

celebrity_file_names_dict

{'lionel_messi': ['./dataset/cropped/lionel_messi\\lionel_messi1.png',
  './dataset/cropped/lionel_messi\\lionel_messi10.png',
  './dataset/cropped/lionel_messi\\lionel_messi11.png',
  './dataset/cropped/lionel_messi\\lionel_messi13.png',
  './dataset/cropped/lionel_messi\\lionel_messi14.png',
  './dataset/cropped/lionel_messi\\lionel_messi15.png',
  './dataset/cropped/lionel_messi\\lionel_messi16.png',
  './dataset/cropped/lionel_messi\\lionel_messi17.png',
  './dataset/cropped/lionel_messi\\lionel_messi18.png',
  './dataset/cropped/lionel_messi\\lionel_messi19.png',
  './dataset/cropped/lionel_messi\\lionel_messi2.png',
  './dataset/cropped/lionel_messi\\lionel_messi20.png',
  './dataset/cropped/lionel_messi\\lionel_messi22.png',
  './dataset/cropped/lionel_messi\\lionel_messi23.png',
  './dataset/cropped/lionel_messi\\lionel_messi24.png',
  './dataset/cropped/lionel_messi\\lionel_messi25.png',
  './dataset/cropped/lionel_messi\\lionel_messi26.png',
  './dataset/cropped/lionel_messi\

In [33]:
def w2d(img, mode='haar', level=1):

    imArray = img

    imArray = cv2.cvtColor(imArray, cv2.COLOR_RGB2GRAY)

    imArray = np.float32(imArray)
    imArray /= 255

    coeffs = pywt.wavedec2(imArray, mode, level=level)

    coeffs_H = list(coeffs)
    coeffs_H[0] *= 0

    imArray_H = pywt.waverec2(coeffs_H, mode)

    imArray_H *= 255
    imArray_H = np.uint8(imArray_H)

    return imArray_H

In [34]:
class_dict = {}
count = 0

for celebrity_name in celebrity_file_names_dict.keys():
    class_dict[celebrity_name] = count
    count += 1

class_dict

{'lionel_messi': 0,
 'maria_sharapova': 1,
 'roger_federer': 2,
 'serena_williams': 3,
 'virat_kohli': 4}

In [35]:
X, y = [], []

for celebrity_name, training_files in celebrity_file_names_dict.items():

    for training_image in training_files:

        img = cv2.imread(training_image)

        if img is None:
            continue

        scaled_raw_img = cv2.resize(img,(32,32))

        img_har = w2d(img,'db1',5)
        scaled_img_har = cv2.resize(img_har,(32,32))

        combined_img = np.vstack((
            scaled_raw_img.reshape(32*32*3,1),
            scaled_img_har.reshape(32*32,1)
        ))

        X.append(combined_img)
        y.append(class_dict[celebrity_name])

In [36]:
X = np.array(X).reshape(len(X),4096).astype(float)
y = np.array(y)

X.shape

(171, 4096)

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=0)

In [42]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', mp['model'])
])

pipe.fit(X_train,y_train)
pipe.score(X_test,y_test)

0.7906976744186046

In [39]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

In [44]:
model_params = {

    'svm': {
        'model': svm.SVC(gamma='auto', probability=True),
        'params': {
            'model__C': [1,10,100,1000],
            'model__kernel': ['rbf','linear']
        }
    },

    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'model__n_estimators': [1,5,10]
        }
    },

    'logistic_regression': {
        'model': LogisticRegression(solver='liblinear', multi_class='auto'),
        'params': {
            'model__C': [1,5,10]
        }
    }

}

In [52]:
scores = []
best_estimators = {}

for algo, mp in model_params.items():

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', mp['model'])
    ])

    clf = GridSearchCV(pipe, mp['params'], cv=5, return_train_score=False)
    clf.fit(X_train, y_train)

    scores.append({
        'model': algo,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

    best_estimators[algo] = clf.best_estimator_

In [47]:
best_clf = best_estimators['svm']

In [48]:
best_clf.score(X_test, y_test)

0.8372093023255814

In [49]:
import joblib
joblib.dump(best_clf, "saved_model.pkl")

['saved_model.pkl']

In [50]:
import json
with open("class_dictionary.json", "w") as f:
    f.write(json.dumps(class_dict))